# 01. Your First Agent

`ArcAgent` is the orchestrator. It owns identity, the tool registry, the module
bus, the context manager, and the session pool — and it delegates the
agentic loop to ArcRun and the LLM call to ArcLLM. The nucleus stays under
3,500 LOC; everything else lives in modules and capabilities.

This notebook stands up an agent end-to-end with mocks — no API keys, no real
LLM calls — and walks the four pillars (identity, sign, authorize, audit) as
they are wired during `startup()`.

You will learn:

- How `ArcAgent` composes arcllm + arcrun + arctrust without mixing concerns.
- Why every `ArcAgent.__init__` requires an `AgentIdentity` (ADR-019).
- The minimum config needed to construct and start an agent.
- How the single streaming entry point `run(input, session=...)` works, and how
  `run_collected(...)` collects it to a one-shot result.
- How steerable runs work via `start_tracked_run()` / `deliver_message()`.
- How tools are registered and visible inside the agent's loop.
- The four-pillar touchpoints in the agent code path.
- The error classes you'll hit when something is misconfigured.


## 1. Setup

The setup cell below is the standard Arc walkthrough preamble: it locates the
repo root, puts every package's `src/` and `tests/` on `sys.path`, and loads
`.env`. This is the **only** way notebooks in this tree wire imports — every
notebook in this directory starts with this exact cell.

In [1]:
# Setup: make Arc packages importable from this notebook
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

_here = Path.cwd()
for _p in [_here, *_here.parents]:
    if (_p / "packages").is_dir() and (_p / "pyproject.toml").is_file():
        REPO_ROOT = _p
        break
else:
    raise RuntimeError("Could not locate Arc repo root")

# Add every package's src/ (and tests/, where present) to sys.path
for _pkg in (REPO_ROOT / "packages").iterdir():
    for _sub in ("src", "tests"):
        _path = _pkg / _sub
        if _path.is_dir() and str(_path) not in sys.path:
            sys.path.insert(0, str(_path))

load_dotenv(REPO_ROOT / ".env")

False

Smoke test — confirm that `arcagent`, `arctrust`, and `arcllm` import cleanly
and that the public symbols this notebook touches are all visible at the top
level of `arcagent`.

In [2]:
import arcagent
from arcagent import (
    ArcAgentError,
    ConfigError,
    ContextError,
    IdentityError,
    ModuleBusError,
    ToolError,
    ToolVetoedError,
)
from arcagent.core.agent import ArcAgent
from arcagent.core.config import (
    AgentConfig,
    ArcAgentConfig,
    IdentityConfig,
    LLMConfig,
    TelemetryConfig,
)
from arcagent.core.errors import IdentityRequired
from arctrust import AgentIdentity

print(f"arcagent {arcagent.__version__}")
print(
    "Public errors:",
    [
        e.__name__
        for e in (
            ArcAgentError,
            ConfigError,
            ContextError,
            IdentityError,
            ModuleBusError,
            ToolError,
            ToolVetoedError,
        )
    ],
)
print("Core type:     ArcAgent")
print("Identity:      AgentIdentity (from arctrust)")

arcagent 0.15.0
Public errors: ['ArcAgentError', 'ConfigError', 'ContextError', 'IdentityError', 'ModuleBusError', 'ToolError', 'ToolVetoedError']
Core type:     ArcAgent
Identity:      AgentIdentity (from arctrust)


## 2. What is ArcAgent?

`ArcAgent` is the **top-level orchestrator** in the Arc stack. It does not
make LLM calls (that's arcllm) and it does not run the agentic loop
(that's arcrun). It binds an identity to a process, owns the security and
observability surface, and routes work to the right component.

The dependency direction is strict and one-way:

```
arctrust  ←  arcllm  ←  arcrun  ←  arcagent
identity     LLM calls   loop       orchestrator (this notebook)
```

`arcagent` imports from all three. Nothing in arctrust, arcllm, or arcrun
imports from arcagent. That's how the nucleus stays under 3,500 LOC: heavy
lifting is delegated, the orchestrator is just wiring.

Here are the components `ArcAgent` owns and the order they're initialized in
during `startup()`. Each one is a separate class and a separate concern.

In [3]:
# Components owned by ArcAgent (wired during startup(), in dependency order)
COMPONENT_ORDER = [
    ("VaultResolver", "optional credential backend (None unless config.vault.backend set)"),
    ("AgentTelemetry", "OpenTelemetry traces, audit events"),
    ("AgentIdentity", "DID + Ed25519 keypair (from arctrust)"),
    ("ModuleBus", "event bus — agent:pre_tool, agent:post_respond, ..."),
    ("PolicyPipeline", "arctrust policy layers — evaluates every tool call"),
    ("ToolRegistry", "register/wrap tools across 4 transports"),
    ("ContextManager", "token budget, prompt assembly, compaction"),
    ("CapabilityRegistry", "unified tools + skills + hooks surface"),
    ("session pool", "SessionManagers created on demand via agent.session(key)"),
]
for i, (name, desc) in enumerate(COMPONENT_ORDER, start=1):
    print(f"  {i}. {name:20s} — {desc}")

  1. VaultResolver        — optional credential backend (None unless config.vault.backend set)
  2. AgentTelemetry       — OpenTelemetry traces, audit events
  3. AgentIdentity        — DID + Ed25519 keypair (from arctrust)
  4. ModuleBus            — event bus — agent:pre_tool, agent:post_respond, ...
  5. PolicyPipeline       — arctrust policy layers — evaluates every tool call
  6. ToolRegistry         — register/wrap tools across 4 transports
  7. ContextManager       — token budget, prompt assembly, compaction
  8. CapabilityRegistry   — unified tools + skills + hooks surface
  9. session pool         — SessionManagers created on demand via agent.session(key)


The orchestrator pattern matters because it pins responsibilities:

- **arcllm** has no concept of an agent. It makes provider-agnostic LLM calls.
- **arcrun** has no concept of identity or audit. It runs a ReAct loop.
- **arctrust** has no concept of a loop or an LLM. It does identity, signing,
  policy, and audit.
- **arcagent** is the only place all four come together.

If a piece of code in `arcagent.core.agent` does an LLM call directly, or
implements its own loop step, it's wrong. Push it down.

## 3. Identity binding

Every `ArcAgent` has exactly one `AgentIdentity` — a DID plus an Ed25519
keypair, sourced from arctrust. Per ADR-019, this is **universal**: personal,
enterprise, and federal tiers all require it. There is no "identity off"
mode. Tier controls *stringency* (FIPS crypto, signed allowlists, max parallel
tools), not *whether* identity exists.

The DID format is `did:arc:<org>:<agent_type>/<hash>` where `<hash>` is the
first 8 bytes of `sha256(public_key)`. Same key in, same DID out.

Notebook `arctrust/01-identity-did.ipynb` is the deep dive — here we'll just
confirm the binding.

In [4]:
# Generate a fresh identity. In production, this is created by
# `arc agent init` once and persisted; the agent loads it on every start.
identity = AgentIdentity.generate(org="acme", agent_type="planner")
print(f"DID:        {identity.did}")
print(f"Can sign:   {identity.can_sign}")
print(f"Pubkey len: {len(identity.public_key)} bytes (Ed25519)")

DID:        did:arc:acme:planner/2dfb6bba
Can sign:   True
Pubkey len: 32 bytes (Ed25519)


Identity is *required* by the orchestrator — but the agent doesn't take an
`AgentIdentity` as a constructor argument. Instead, `ArcAgent` is configured
with an `IdentityConfig` (`did`, `key_dir`, `vault_path`) and resolves the
identity inside `startup()` via `AgentIdentity.from_config(...)`.

If `config.identity.did` is empty, `from_config` generates a new keypair, saves
it, and writes the DID back into the config file. If it's set, it loads the
keypair from disk (or from a vault, if configured).

In [5]:
# Inspect the config sub-model that drives identity resolution.
ic = IdentityConfig(did="", key_dir="~/.arcagent/keys", vault_path="")
print("did       :", repr(ic.did), "  (empty → generate-and-persist on first startup)")
print("key_dir   :", ic.key_dir)
print("vault_path:", repr(ic.vault_path), "  (empty → file-backed; non-empty → vault-backed)")

did       : ''   (empty → generate-and-persist on first startup)
key_dir   : ~/.arcagent/keys
vault_path: ''   (empty → file-backed; non-empty → vault-backed)


## 4. Constructing an agent

The minimum viable `ArcAgentConfig` needs:

- `agent.name` — human-readable label, used in logs and trace attribution.
- `llm.model` — `provider/model` string passed to ArcLLM (e.g.
  `anthropic/claude-sonnet-4-6`). We'll use `test/model` so the mock path
  is obvious.
- `identity.key_dir` — where to persist the generated keypair.

Everything else has a default. We'll point everything at a `tmp_path` so this
notebook leaves no side effects on disk.

In [6]:
import tempfile

# All paths under one tmp dir so we leave nothing behind.
tmpdir = tempfile.TemporaryDirectory()
TMP = Path(tmpdir.name)

config = ArcAgentConfig(
    agent=AgentConfig(
        name="first-agent",
        org="acme",
        type="planner",
        workspace=str(TMP / "workspace"),
    ),
    llm=LLMConfig(model="test/model"),
    identity=IdentityConfig(
        did="",  # empty → generate
        key_dir=str(TMP / "keys"),
        vault_path="",
    ),
    telemetry=TelemetryConfig(enabled=False),  # quiet for the notebook
)
print("Config built. Workspace:", config.agent.workspace)
print("Model spec     :", config.llm.model)

Config built. Workspace: /var/folders/tc/283s9y5x2ws8rtq0c9skdn700000gn/T/tmp7wqy6n6x/workspace
Model spec     : test/model


`ArcAgent.__init__` is **cheap** — it stores config, resolves the workspace
path, and creates a reload lock. None of the components (telemetry, identity,
bus, registries) exist yet. They're created in `startup()`.

In [7]:
agent = ArcAgent(config=config)
print("Agent created.")
print("  workspace:    ", agent._workspace)
print("  started?      ", agent._started)
print("  bus?          ", agent._bus)
print("  identity?     ", agent._identity)

Agent created.
  workspace:     /private/var/folders/tc/283s9y5x2ws8rtq0c9skdn700000gn/T/tmp7wqy6n6x/workspace
  started?       False
  bus?           None
  identity?      None


Now call `startup()`. This walks the component initialization, generates
the identity (because `did=""`), and emits two bus events on the way out:
`agent:ready` (carrying the capability registry for late subscribers) and
`agent:init` (the lifecycle marker).

In [8]:
import asyncio

await agent.startup()

print("Started:    ", agent._started)
print("Identity:   ", agent._identity.did)
print("Bus:        ", type(agent._bus).__name__)
print("Telemetry:  ", type(agent._telemetry).__name__)
print("Tool reg:   ", type(agent._tool_registry).__name__)
print("Context:    ", type(agent._context).__name__)
print("Sessions:   ", f"{len(agent._sessions)} open (created on demand via agent.session(key))")

Cannot read config to persist DID: arcagent.toml


Started:     True
Identity:    did:arc:acme:planner/11146e17
Bus:         ModuleBus
Telemetry:   AgentTelemetry
Tool reg:    ToolRegistry
Context:     ContextManager
Sessions:    0 open (created on demand via agent.session(key))


Notice the DID was generated automatically (we passed `did=""`) and persisted
to `key_dir`. A second startup with the same config would *load* this same
identity — this is what makes agent identity stable across restarts.

In [9]:
key_files = sorted((TMP / "keys").rglob("*"))
print(f"Persisted {len(key_files)} key file(s) under {TMP / 'keys'}:")
for f in key_files:
    if f.is_file():
        print("  -", f.relative_to(TMP / "keys"), f"({f.stat().st_size} bytes)")

Persisted 2 key file(s) under /var/folders/tc/283s9y5x2ws8rtq0c9skdn700000gn/T/tmp7wqy6n6x/keys:
  - did_arc_acme_planner_11146e17.key (32 bytes)
  - did_arc_acme_planner_11146e17.pub (32 bytes)


## 5. The first run

`agent.run_collected(task, session_key=...)` is the one-shot entry point:
open-or-resume the keyed session, stream the turn through `agent.run()`
(which assembles the system prompt, snapshots the registered tools, emits
`agent:pre_respond`, drives `arcrun_run_stream`, emits `agent:post_respond`),
and collect the stream to a final `RunResult`.

We don't have a real LLM, so we patch `load_eval_model` (the model loader,
imported into `arcagent.core.model_manager`) and `arcrun_run_stream` (the
loop entry point, imported into `arcagent.core.agent_dispatch`) with mocks.
This is the *exact* pattern used in `tests/unit/core/test_agent_run.py` —
the agent code path is unchanged; only the leaves are stubbed.

In [10]:
from collections.abc import AsyncIterator
from unittest.mock import AsyncMock, MagicMock, patch

from arcrun import StreamEvent, TurnEndEvent

# Stub model — never actually called because arcrun_run_stream is mocked too.
mock_model = MagicMock(name="StubModel")
mock_model.close = AsyncMock()  # closed during shutdown()


async def _fake_stream() -> AsyncIterator[StreamEvent]:
    yield TurnEndEvent(final_text="Mock model says: 2 + 2 = 4", turns=1, tool_calls_made=0)


async def _fake_run_stream(*args, **kwargs) -> AsyncIterator[StreamEvent]:
    return _fake_stream()


with (
    patch("arcagent.core.model_manager.load_eval_model", return_value=mock_model),
    patch(
        "arcagent.core.agent_dispatch.arcrun_run_stream", side_effect=_fake_run_stream
    ) as mock_run_stream,
):
    result = await agent.run_collected("What is 2+2?", session_key="demo")

    print("Result content          :", result.content)
    print("Tool calls made         :", result.tool_calls_made)
    print("arcrun_run_stream calls :", mock_run_stream.call_count)
    # Inspect the kwargs the agent passed down to arcrun.
    call_kwargs = mock_run_stream.call_args.kwargs
    print("arcrun_run_stream kwargs:", sorted(call_kwargs))

Result content          : Mock model says: 2 + 2 = 4
Tool calls made         : 0
arcrun_run_stream calls : 1
arcrun_run_stream kwargs: ['actor_did', 'approval_provider', 'approval_required_tools', 'capabilities', 'max_consecutive_errors', 'max_cost_usd', 'max_parallel', 'max_repeat', 'max_tokens', 'messages', 'model', 'on_checkpoint', 'on_event', 'store_raw_bodies', 'system_prompt', 'task', 'tool_choice', 'transform_context']


Every value in those kwargs comes from the agent's wiring — the system prompt
assembled by `ContextManager`, the tool list snapshot from `ToolRegistry`, the
event bridge produced by `create_arcrun_bridge`. The agent never inspects what
the model says or rewrites the loop; it sets the table and steps back.

`run_collected()` is session-bound, not fire-and-forget: every call appends to
the same session's history — a per-session JSONL file under
`<workspace>/sessions/`. With the same patches in place, two `run_collected()`
calls against the same `session_key` share history.

In [11]:
with (
    patch("arcagent.core.model_manager.load_eval_model", return_value=mock_model),
    patch("arcagent.core.agent_dispatch.arcrun_run_stream", side_effect=_fake_run_stream),
):
    await agent.run_collected("Hi, I'm Alice.", session_key="demo")
    await agent.run_collected("What's my name?", session_key="demo")

    session = await agent.session("demo")
    msgs = session.get_messages()
    print("Session id        :", session.session_id)
    print("Messages persisted:", len(msgs))
    for m in msgs:
        content = m["content"]
        print(f"  {m['role']:9s} | {content[:60]}")

Session id        : demo
Messages persisted: 6
  user      | What is 2+2?
  assistant | Mock model says: 2 + 2 = 4
  user      | Hi, I'm Alice.
  assistant | Mock model says: 2 + 2 = 4
  user      | What's my name?
  assistant | Mock model says: 2 + 2 = 4


The `JSONL` file lands under the workspace, partitioned by session id:

In [12]:
sessions_dir = Path(agent._config.agent.workspace) / "sessions"
for f in sorted(sessions_dir.glob("*.jsonl")):
    print(f"  {f.name}  ({f.stat().st_size} bytes)")

  demo.jsonl  (734 bytes)


Need a non-blocking handle so you can steer or cancel mid-run? Use
`start_tracked_run(task, session_key=...)`. It drives `arcrun.run_async` and
returns arcrun's own **`RunHandle`** directly — there's no agent-layer
wrapper. The handle is also tracked internally under `session_key`, so
`agent.active_run(session_key)` and `agent.deliver_message(...)` can reach
the same live run from another caller.

In [13]:
from arcrun import RunHandle

mock_handle = AsyncMock(spec=RunHandle)
mock_handle.result = AsyncMock(
    return_value=MagicMock(content="Mock model says: done", tool_calls_made=0)
)
mock_handle.state = MagicMock(name="RunState")

with (
    patch("arcagent.core.model_manager.load_eval_model", return_value=mock_model),
    patch(
        "arcagent.core.agent_dispatch.arcrun_run_async", new=AsyncMock(return_value=mock_handle)
    ),
):
    handle = await agent.start_tracked_run("Long-running task", session_key="tracked")

    print("Type              :", type(handle).__name__)
    print("Is the mock handle?", handle is mock_handle)
    print("Has steer?        :", callable(handle.steer))
    print("Has follow_up?    :", callable(handle.follow_up))
    print("Has cancel?       :", callable(handle.cancel))
    print("Tracked as active?", agent.active_run("tracked") is handle)

    # Drive it like a real run — steer, then complete. RunHandle.steer takes
    # the caller's DID (arcrun records it; authorization is the caller's job).
    await handle.steer(agent._identity.did, "Actually focus on X instead")
    result = await handle.result()
    print("Final             :", result.content)

Type              : AsyncMock
Is the mock handle? True
Has steer?        : True
Has follow_up?    : True
Has cancel?       : True
Tracked as active? True
Final             : Mock model says: done


`RunHandle.result()` just `await`s the underlying `asyncio.Task`, so it's
safe to call more than once (a completed task returns its cached result).
The agent's own finalizer awaits it exactly once in the background — the
moment the loop finishes, it commits the assistant turn, runs compaction,
and drops the handle from `agent._active_runs` so `active_run(session_key)`
goes back to `None`.

## 6. Basic tool registration

The `ToolRegistry` is the agent's view of "what tools can the LLM call?". Every
registered tool is wrapped with policy evaluation, audit emission, telemetry
spans, and pre/post bus events before its `execute` is invoked. Capability
loaders (skills, modules, builtins) feed this registry; you can also register
tools manually for tests and quick experiments.

Here we'll register a synthetic `add_numbers` tool with `NATIVE` transport
(in-process Python) and confirm it shows up in `to_arcrun_tools()` — that's
the snapshot the agent hands to arcrun on every loop entry.

In [14]:
from arcagent.core.tool_registry import RegisteredTool, ToolTransport


async def _add_execute(a: int, b: int) -> dict:
    """Synthetic tool body — no I/O, no side effects."""
    return {"sum": a + b}


add_tool = RegisteredTool(
    name="add_numbers",
    description="Add two integers and return the sum.",
    input_schema={
        "type": "object",
        "properties": {
            "a": {"type": "integer"},
            "b": {"type": "integer"},
        },
        "required": ["a", "b"],
    },
    transport=ToolTransport.NATIVE,
    execute=_add_execute,
    classification="read_only",  # no state mutation → eligible for parallel batches
    capability_tags=["arithmetic"],
    source="walkthrough/01-first-agent",
)

agent._tool_registry.register(add_tool)
print(f"Registered: {add_tool.name} ({add_tool.transport.value})")

Registered: add_numbers (native)


The registry exposes the tool list to the loop via `to_arcrun_tools()` —
that's the object the agent passes into `arcrun.run` as `tools=...`. Capability
tools registered by the loader during `startup()` show up here too, so the
LLM sees one merged toolset.

In [15]:
arcrun_tools = agent._tool_registry.to_arcrun_tools()
print(f"Total tools available to the loop: {len(arcrun_tools)}")
for t in arcrun_tools:
    if t.name == "add_numbers":
        print(f"  -> {t.name}: {t.description}")
        break
else:
    print("  (add_numbers not found — registry view diverged)")

Total tools available to the loop: 13
  -> add_numbers: Add two integers and return the sum.


Unregistering is symmetric. Real reloads (`agent.reload()`) drop and
re-register capability tools wholesale; manual registrations are usually
torn down by the test that created them.

In [16]:
removed = agent._tool_registry.unregister("add_numbers")
print("Unregistered :", removed)

arcrun_tools_after = agent._tool_registry.to_arcrun_tools()
names_after = {t.name for t in arcrun_tools_after}
print("add_numbers still in toolset?", "add_numbers" in names_after)

Unregistered : True
add_numbers still in toolset? False


## 7. Errors you'll hit

Every error in `arcagent.core.errors` carries a structured `code`,
`component`, `message`, and `details` dict — designed to flow into the audit
trail without losing context. The base hierarchy:

```
ArcAgentError                 — base for everything below
├── ConfigError               — TOML or Pydantic validation failure
├── IdentityError             — DID parse, key load, signing failure
│   └── IdentityRequired      — DID missing in config
├── ToolError                 — tool execution failure
│   └── ToolVetoedError       — pre_tool handler vetoed the call
├── ContextError              — token budget / prompt assembly
└── ModuleBusError            — handler timeout / lifecycle failure
```

The next few cells trip each one intentionally and catch it, printing the
structured fields. Each cell completes successfully — *raising the expected
error is the success criterion*.

In [17]:
# ConfigError — vault backend reference must be `module:Class`. A bare
# string with no colon is rejected at startup, before any module loads.
from arcagent.core.config import VaultConfig

try:
    bad_cfg = ArcAgentConfig(
        agent=AgentConfig(name="bad-vault", workspace=str(TMP / "ws_bad")),
        llm=LLMConfig(model="test/model"),
        identity=IdentityConfig(key_dir=str(TMP / "keys_bad")),
        vault=VaultConfig(backend="this_has_no_colon"),
        telemetry=TelemetryConfig(enabled=False),
    )
    bad_agent = ArcAgent(config=bad_cfg)
    await bad_agent.startup()
except ConfigError as e:
    print(f"ConfigError caught: code={e.code}, component={e.component}")
    print(f"  message: {e.message}")
    print(f"  details: {e.details}")

ConfigError caught: code=CONFIG_INVALID_VAULT_BACKEND, component=config
  message: Invalid vault backend format (missing ':'): this_has_no_colon
  details: {'backend': 'this_has_no_colon'}


In [18]:
# IdentityError (specifically IdentityRequired) — surface the structured
# hint the CLI emits when the agent has no DID configured. The base
# resolver only raises this when *every* fallback path is unavailable;
# here we instantiate the error directly to inspect its shape.
ir = IdentityRequired()
print(f"  Subclass of IdentityError? {isinstance(ir, IdentityError)}")
print(f"  Subclass of ArcAgentError? {isinstance(ir, ArcAgentError)}")
print(f"  code     : {ir.code}")
print(f"  component: {ir.component}")
print(f"  message  : {ir.message}")
print(f"  details  : {ir.details}")

  Subclass of IdentityError? True
  Subclass of ArcAgentError? True
  code     : IDENTITY_REQUIRED
  component: identity
  message  : Agent DID is required. Run 'arc agent init' to generate a DID and keypair.
  details  : {'hint': 'arc agent init'}


In [19]:
# IdentityError — DID set in config but the key file does not exist.
# Per ADR-019 there is no silent-regen path; the agent must fail loud.
try:
    bogus_cfg = ArcAgentConfig(
        agent=AgentConfig(name="ghost", workspace=str(TMP / "ws_ghost")),
        llm=LLMConfig(model="test/model"),
        identity=IdentityConfig(
            did="did:arc:acme:planner/deadbeef",  # claims a DID...
            key_dir="/nonexistent/key/dir",  # ...but no key on disk
        ),
        telemetry=TelemetryConfig(enabled=False),
    )
    ghost = ArcAgent(config=bogus_cfg)
    await ghost.startup()
except (IdentityError, ValueError) as e:
    print(f"Caught: {type(e).__name__}: {e}")

Caught: ValueError: Key file not found: /nonexistent/key/dir/did_arc_acme_planner_deadbeef.key


In [20]:
# ContextError — exists for token-budget and prompt-assembly failures.
# We can't easily trip it with mocks (it requires a real prompt over budget),
# so just confirm the class is wired correctly and inspect its shape.
ctx_err = ContextError(
    code="CONTEXT_BUDGET_EXCEEDED",
    message="Prompt would exceed 128k tokens after compaction",
    details={"requested": 132_000, "budget": 128_000},
)
print(f"Type:       {type(ctx_err).__name__}")
print(f"Component:  {ctx_err.component}")
print(f"Code:       {ctx_err.code}")
print(f"Subclass of ArcAgentError? {isinstance(ctx_err, ArcAgentError)}")
print(str(ctx_err))

Type:       ContextError
Component:  context_manager
Code:       CONTEXT_BUDGET_EXCEEDED
Subclass of ArcAgentError? True
[CONTEXT_BUDGET_EXCEEDED] context_manager: Prompt would exceed 128k tokens after compaction


In [21]:
# ToolError / ToolVetoedError — emitted by ToolRegistry when a tool call
# fails or a pre_tool hook denies it. We construct one directly to show
# the shape — the registry raises these at runtime.
veto = ToolVetoedError("write_file vetoed by sandbox policy", details={"path": "/etc/passwd"})
print(f"Type:               {type(veto).__name__}")
print(f"Subclass ToolError? {isinstance(veto, ToolError)}")
print(f"code:               {veto.code}")
print(f"details:            {veto.details}")
print(str(veto))

Type:               ToolVetoedError
Subclass ToolError? True
code:               TOOL_VETOED
details:            {'path': '/etc/passwd'}
[TOOL_VETOED] tool_registry: write_file vetoed by sandbox policy


In [22]:
# ModuleBusError — emitted when a bus handler times out or its lifecycle
# fails. This is intentionally a thin wrapper class; instantiate to confirm.
mbe = ModuleBusError(
    code="MODULE_HANDLER_TIMEOUT",
    message="agent:post_tool handler exceeded 5s budget",
    details={"event": "agent:post_tool", "handler": "audit_module"},
)
print(f"  component: {mbe.component}")
print(f"  code     : {mbe.code}")
print(str(mbe))

  component: module_bus
  code     : MODULE_HANDLER_TIMEOUT
[MODULE_HANDLER_TIMEOUT] module_bus: agent:post_tool handler exceeded 5s budget


## 8. The four pillars in this code path

ADR-019 says every Arc deployment, at every tier, enforces:

1. **Identity** — every entity has a DID; every tool dispatch carries `caller_did`.
2. **Sign**     — every loaded artifact (skill, extension, module) is verified.
3. **Authorize** — `arctrust.policy.PolicyPipeline` evaluates every tool call,
   first-DENY-wins, fail-closed.
4. **Audit**    — `arctrust.audit.emit(AuditEvent, sink)` on every operation.

Tier (`personal` / `enterprise` / `federal`) is **stringency metadata, not a
gate**. The pillars apply universally; what changes is FIPS-validated crypto,
signed allowlists, max parallel tools, and the number of policy layers.

Let's walk where each touches the agent we just built.

In [23]:
# Pillar 1: Identity — bound at startup(), readable on the agent.
print("=" * 60)
print("Pillar 1: IDENTITY")
print("=" * 60)
print(f"  agent DID         : {agent._identity.did}")
print(f"  can sign?         : {agent._identity.can_sign}")
print(f"  pubkey (hex)      : {agent._identity.public_key.hex()[:32]}...")
print(f"  tier (stringency) : {agent._config.security.tier}")

Pillar 1: IDENTITY
  agent DID         : did:arc:acme:planner/11146e17
  can sign?         : True
  pubkey (hex)      : be6bd92dc20c9d7e82c22a1d81f16227...
  tier (stringency) : personal


In [24]:
# Pillar 2: Sign — every capability is verified before load.
# The capability registry holds the result of that verification.
# (We registered no signed bundles in this notebook, so the count is 0.)
print("=" * 60)
print("Pillar 2: SIGN — verified capability bundles")
print("=" * 60)
reg = agent._capability_registry
if reg is None:
    print("  capability registry not initialized")
else:
    print(f"  tools  registered: {len(reg._tools)}")
    print(f"  skills registered: {len(reg._skills)}")
    print(f"  hooks  registered: {sum(len(h) for h in reg._hooks.values())}")
    print("  (every entry above passed AST validation + signature checks)")

Pillar 2: SIGN — verified capability bundles
  tools  registered: 12
  skills registered: 4
  hooks  registered: 0
  (every entry above passed AST validation + signature checks)


In [25]:
# Pillar 3: Authorize — every tool dispatch flows through the policy pipeline.
# ToolRegistry holds the pipeline; the pipeline's layer count depends on tier.
print("=" * 60)
print("Pillar 3: AUTHORIZE — policy pipeline")
print("=" * 60)
tr = agent._tool_registry
pipeline = tr._policy_pipeline if hasattr(tr, "_policy_pipeline") else None
if pipeline is not None:
    layers = [type(layer).__name__ for layer in getattr(pipeline, "_layers", [])]
    print(f"  layers ({len(layers)}): {layers}")
else:
    print("  pipeline attached to registry: instance ready, layers tier-driven")
print(f"  tier-config layer count is set by config.security.tier='{agent._config.security.tier}'")

Pillar 3: AUTHORIZE — policy pipeline
  layers (2): ['IdentityLayer', 'GlobalLayer']
  tier-config layer count is set by config.security.tier='personal'


In [26]:
# Pillar 4: Audit — telemetry.audit_event() is the agent's hook into
# arctrust's emit(). A denied mid-turn steer is the audited action reachable
# from this notebook: `deliver_message` → `_authorize_steer` fails CLOSED on
# ANY pipeline exception, and every denial is audited (REQ-041). We'll spy on
# audit_event and force that path by making the policy pipeline raise.
print("=" * 60)
print("Pillar 4: AUDIT")
print("=" * 60)

audit_log: list[tuple[str, dict]] = []
original = agent._telemetry.audit_event


def _spy(event_name: str, fields: dict) -> None:
    audit_log.append((event_name, fields))
    return original(event_name, fields)


agent._telemetry.audit_event = _spy  # type: ignore[assignment]

with patch.object(agent._policy_pipeline, "evaluate", side_effect=RuntimeError("boom")):
    authorized = await agent._authorize_steer(agent._identity.did)

agent._telemetry.audit_event = original  # restore

print("Steer authorized?     :", authorized, "(False — fail-closed on a raising pipeline)")
print(f"  audit events captured: {len(audit_log)}")
for name, fields in audit_log:
    keys = sorted(fields)
    print(f"   - {name:24s} fields={keys}")

Pillar 4: AUDIT
Steer authorized?     : False (False — fail-closed on a raising pipeline)
  audit events captured: 1
   - messaging.steer.denied   fields=['caller_did', 'layer', 'reason', 'rule_id']


Every one of those events would, in production, emit through
`arctrust.audit.emit(AuditEvent, sink)` — `JsonlSink` for compliance,
`SignedChainSink` for tamper-evident chain, and `arcui.bridge.UIBridgeSink`
for the live dashboard. A single emission point with sinks fanning out is
the only thing that prevents schema drift across modules.

## 9. Cleanup

`agent.shutdown()` is the symmetric counterpart to `startup()`. It walks the
component teardown in reverse: emits `agent:shutdown`, drains capability
lifecycles, closes the LLM `httpx` client (if a model was loaded), and clears
internal state. Idempotent — calling it twice is safe.

In [27]:
await agent.shutdown()
print("Started after shutdown:", agent._started)

# Idempotent — second call is a no-op.
await agent.shutdown()
print("Still safe to call again.")

# Clean up the tmp dir we created at the top.
tmpdir.cleanup()
print("Workspace tmp dir removed.")

Started after shutdown: False
Still safe to call again.
Workspace tmp dir removed.


## 10. Summary

You stood up a real `ArcAgent` end-to-end with mocks, walked the four
pillars, and saw where each public symbol fits.

**What you used:**

- `ArcAgent` — the orchestrator class. `__init__` is cheap; `startup()` wires
  9 components in dependency order; `shutdown()` reverses them.
- `agent.run(input, session=...)` — the single streaming entry point.
  `agent.run_collected(input, session_key=...)` opens-or-resumes the keyed
  session and collects the stream to a `RunResult`.
- `agent.start_tracked_run(input, session_key=...)` — the non-blocking,
  steerable entry point. Returns arcrun's own `RunHandle` directly (no
  agent-layer wrapper), tracked under `session_key`. Exposes
  `steer(caller_did, message)`, `follow_up(caller_did, message)`, `cancel()`,
  `result()`, `state`. `agent.active_run(session_key)` and
  `agent.deliver_message(...)` reach the same tracked handle.
- `AgentIdentity` — DID + Ed25519 keypair from arctrust. Required by every
  agent at every tier (ADR-019). Resolved from `IdentityConfig` in `startup()`.
- `ArcAgentConfig` + `AgentConfig` + `LLMConfig` + `IdentityConfig` +
  `TelemetryConfig` — the minimal config model tree.
- `RegisteredTool` + `ToolTransport` — manual tool registration into
  `ToolRegistry`.
- The error hierarchy: `ArcAgentError`, `ConfigError`, `IdentityError`,
  `IdentityRequired`, `ToolError`, `ToolVetoedError`, `ContextError`,
  `ModuleBusError`.

**The four pillars in code:**

| Pillar    | Where it lives                                              |
|-----------|-------------------------------------------------------------|
| Identity  | `AgentIdentity.from_config(...)` in `ArcAgent.startup()`    |
| Sign      | `CapabilityLoader` AST-validates + signature-checks bundles |
| Authorize | `arctrust.policy.PolicyPipeline` in `ToolRegistry.dispatch` |
| Audit     | `AgentTelemetry.audit_event` → `arctrust.audit.emit`        |

**Concerns that are deliberately not here:**

- LLM provider details — that's `arcllm`.
- Loop steps, ReAct strategy, tool execution mechanics — that's `arcrun`.
- Identity primitives (DID format, Ed25519 sign/verify, child derivation) —
  that's `arctrust`.

**Next:** see `walkthroughs/arcagent/02-tool-integration.ipynb` — the four
tool transports (in-process, subprocess, MCP, HTTP) and what `ToolError` /
`ToolVetoedError` look like at runtime.